In [3]:
import json
import pandas as pd
import requests
import time
import re
import multiprocessing
import math
from transformers import BertModel, BertConfig, BertTokenizer
import torch,gc
from collections import Counter

In [5]:
comment_data = []

# 打开JSON文件
with open('/remote-home/cs_acmis_wsf/ai4dingo/mooccubex/comment.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        comment = json.loads(line)
        Id = comment["id"]
        user_id = comment["user_id"]
        text = comment["text"].replace(' ', '').replace('\t', '').replace('\n', '')
        create_time = comment["create_time"]
        comment_data.append([Id,user_id,text,create_time])

### 数据清理

In [7]:
df_comment = pd.DataFrame(comment_data,columns=["id","user_id","text","create_time"])

In [8]:
### 删除 text 列中字符串数少于10个字的行
df_comment['text_length'] = df_comment['text'].str.len()

df_comment = df_comment[df_comment['text_length'] >= 10]
df_comment = df_comment[df_comment['text_length'] <= 128]

df_comment = df_comment.drop('text_length', axis=1)


In [9]:
### 删除text列数据重复频率大于1的行

df_comment = df_comment.drop_duplicates(subset='text', keep=False)


In [10]:
### 清洗掉 text 中的序号
text = df_comment['text']
pattern2 = r'^\s*\d+[\u3001、]'
# 使用str.replace方法替换匹配到的部分为空字符串
text_cleaned = text.str.replace(pattern2, '', regex=True)
df_comment['text'] = text_cleaned

In [11]:
### 清洗掉 text 中的序号
text = df_comment['text']
pattern3 = r'^\s*\d+[\u3001.]'
# 使用str.replace方法替换匹配到的部分为空字符串
text_cleaned = text.str.replace(pattern3, '', regex=True)
df_comment['text'] = text_cleaned

### 计算最大字符串

In [12]:
text = df_comment["text"].tolist()

In [13]:
max_len = max(len(t) for t in text)

In [14]:
max_len

128

### 文本转向量

In [15]:
def bert_define(path="/remote-home/cs_acmis_wsf/ai4dingo/bert_chinese_pretrained"):
    # 加载bert的tokenizer分词
    tokenizer = BertTokenizer.from_pretrained(path)
    # 加载预训练模型
    model_config = BertConfig.from_pretrained(path)
    model = BertModel.from_pretrained(path, config=model_config)
    model = model.cuda()
    return tokenizer, model

In [16]:
# 构成一个小batch
batch_token, batch_segment, batch_mask = list(), list(), list()
tokenizer,model = bert_define()

Some weights of the model checkpoint at /remote-home/cs_acmis_wsf/ai4dingo/bert_chinese_pretrained were not used when initializing BertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [17]:
for i in range(len(text)):
    t = text[i]
    if i % 100000 == 0 and i != 0:  # 检查是否是10000的倍数且不是第0次迭代
        print(f"处理到第 {(i // 100000)} 个10W")
     
    # text 作 tokenizer 分词
    token = tokenizer.tokenize(t)
    token = ['[CLS]'] + token + ['[SEP]']
    token_id = tokenizer.convert_tokens_to_ids(token)  # 字转换vocab中的index

    # 加padding补齐及segment、mask
    padding = [0] * (max_len - len(token_id))
    mask = [1] * len(token_id) + padding
    segment = [0] * len(token_id) + padding
    token_id = token_id + padding

    batch_token.append(token_id)
    batch_segment.append(segment)
    batch_mask.append(mask)

print("======文本处理完毕======")    

处理到第 1 个10W
处理到第 2 个10W
处理到第 3 个10W
处理到第 4 个10W
处理到第 5 个10W
处理到第 6 个10W
处理到第 7 个10W
处理到第 8 个10W
处理到第 9 个10W
处理到第 10 个10W
处理到第 11 个10W
处理到第 12 个10W
处理到第 13 个10W
======文本处理完毕======


In [18]:
dim  = [len(d) for d in batch_token]

#### 清除 dim > max_len的数据

In [19]:
err_dim = []
for i in range(len(dim)):
    if dim[i] != max_len:
        err_dim.append(i)

In [20]:
batch_token_new  = []
for bt in batch_token:
    if len(bt) == max_len:
        batch_token_new.append(bt)
        
        
batch_segment_new = []
for bs in batch_segment:
    if len(bs) == max_len:
        batch_segment_new.append(bs)
        
batch_mask_new = []
for bm in batch_mask:
    if len(bm) == max_len:
        batch_mask_new.append(bm)

In [21]:
df_comment = df_comment.reset_index(drop=True)

In [22]:
df_comment = df_comment.drop(err_dim)

In [24]:
df_comment = df_comment.reset_index(drop=True)

In [30]:
vector_count = min([len(df_comment),len(batch_mask),len(batch_segment),len(batch_token)])

#### 计算最近best_n, best_batch_size, best_batch

In [31]:
def closest_batch_size(vector_count, min_n=6, max_n=12):
    best_n = min_n
    best_batch_size = 2**best_n
    best_batch = vector_count // best_batch_size
    min_difference = abs(vector_count - best_batch * best_batch_size)

    for n in range(min_n + 1, max_n + 1):
        batch_size = 2**n
        batch = vector_count // batch_size
        difference = abs(vector_count - batch * batch_size)

        # 检查当前batch_size是否比之前的更接近vector_count
        if difference < min_difference:
            best_n = n
            best_batch_size = batch_size
            best_batch = batch
            min_difference = difference

    return best_n, best_batch_size, best_batch

In [32]:
n, batch_size, batch = closest_batch_size(vector_count)
print(f"vector_count: {vector_count}, n: {n}, batch_size: {batch_size}, batch: {batch}")

vector_count: 1368559, n: 6, batch_size: 64, batch: 21383


#### 计算 max_vector_count

In [47]:
max_vector_count = 2**n * batch

In [34]:
df_comment = df_comment.head(max_vector_count)

In [36]:
df_comment.to_csv(f'/remote-home/cs_acmis_wsf/ai4dingo/handle_data/csv/comment_max_vector.csv', index=False)

#### 分成40份文件来存储 以防止处理时显存溢出

In [50]:
max_vector_count

1368512

In [74]:
def split_dataframe(df, n_parts, rows_per_part):
    # 计算总行数
    total_rows = len(df)
    # 计算前n-1份的总行数
    rows_for_first_n_minus_one_parts = (n_parts - 1) * rows_per_part
    # 确定是否需要额外的部分来存储剩余的数据
    extra_part_needed = total_rows > rows_for_first_n_minus_one_parts

    # 创建一个包含所有分片的列表
    dfs = []

    # 分配前n-1份
    for i in range(n_parts - 1):
        start = i * rows_per_part
        end = start + rows_per_part
        dfs.append(df.iloc[start:end])

    # 如果需要，分配最后一部分
    if extra_part_needed:
        start = (n_parts - 1) * rows_per_part
        dfs.append(df.iloc[start:])

    return dfs


In [75]:
n = 40
rows_per_part = 34816  # 每份的行数
split_dfs_comment = split_dataframe(df_comment, n,rows_per_part)

In [81]:
for i in range(len(split_dfs_comment)):
    df = split_dfs_comment[i]
    df.to_csv(f'/remote-home/cs_acmis_wsf/ai4dingo/handle_data/csv/comment/chunk_{i+1}.csv', index=False)

### 循环处理

In [110]:
batch_size_num = 2048

# for i in range(len(split_dfs_comment)-1):
for i in range(len(split_dfs_comment)):
    df = split_dfs_comment[i]
    print(f'正在读取第 {i+1} 个DF')
    # df = pd.read_csv(f'/remote-home/cs_acmis_wsf/ai4dingo/handle_data/csv/comment/chunk_{i+1}.csv')
    text = df["text"].tolist()
    
    print(f'初始化batch数据： {i+1}')
    batch_token, batch_segment, batch_mask = list(), list(), list()
    for t in text:
        # text 作 tokenizer 分词
        token = tokenizer.tokenize(str(t))
        token = ['[CLS]'] + token + ['[SEP]']
        token_id = tokenizer.convert_tokens_to_ids(token)  # 字转换vocab中的index

        # 加padding补齐及segment、mask
        padding = [0] * (max_len - len(token_id))
        mask = [1] * len(token_id) + padding
        segment = [0] * len(token_id) + padding
        token_id = token_id + padding

        batch_token.append(token_id)
        batch_segment.append(segment)
        batch_mask.append(mask)
        
    print(f'batch数据 组装完毕 ：{i+1}')
   
    batch_tensor_token = torch.tensor(batch_token)
    batch_tensor_segment = torch.tensor(batch_segment)
    batch_tensor_mask = torch.tensor(batch_mask)

    batch_len = len(batch_tensor_token)
    
    if torch.cuda.is_available():
        batch_tensor_token = batch_tensor_token.to('cuda:0')
        batch_tensor_segment = batch_tensor_segment.to('cuda:0')
        batch_tensor_mask = batch_tensor_mask.to('cuda:0')
    
    print(f'batch_tensor数据 组装完毕 ：{i+1},开始分块处理')
    
    batch_tensor_token_chunked = [batch_tensor_token[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    batch_tensor_segment_chunked = [batch_tensor_segment[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    batch_tensor_mask_chunked = [batch_tensor_mask[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    print(f'分块处理完毕数据完毕，每块的长度为:{batch_size_num},一共有 {len(batch_tensor_token_chunked)} 块， 开始调用模型')
    
    gc.collect()
    torch.cuda.empty_cache()
    text_vector = []
    
    try:
        for k in range(len(batch_tensor_token_chunked)):
            with torch.no_grad():
                print(f'正在处理 {i+1} ----- {k+1}')
                outputs = model(batch_tensor_token_chunked[k], token_type_ids=batch_tensor_segment_chunked[k], attention_mask=batch_tensor_mask_chunked[k])
                outputs = outputs[0][:, 0, :]  # 取cls向量
                for o in range(outputs.shape[0]):
                    text_vector.append(outputs[o])
        df["text_tersor"] = [ tv.tolist() for tv in  text_vector] 
        df.to_csv(f'/remote-home/cs_acmis_wsf/ai4dingo/handle_data/csv/comment/tensor_{i+1}.csv', index=False)
        print(f"文件写入完毕 {i+1}")
                    
    except Exception as e:
        print(f"出现异常 {i+1}: {e}")
        continue
            
    
    print('============================')
    print()

正在读取第 1 个DF
初始化batch数据： 1
batch数据 组装完毕 ：1
batch_tensor数据 组装完毕 ：1,开始分块处理
分块处理完毕数据完毕，每块的长度为:2048,一共有 17 块， 开始调用模型
正在处理 1 ----- 1
正在处理 1 ----- 2
正在处理 1 ----- 3
正在处理 1 ----- 4
正在处理 1 ----- 5
正在处理 1 ----- 6
正在处理 1 ----- 7
正在处理 1 ----- 8
正在处理 1 ----- 9
正在处理 1 ----- 10
正在处理 1 ----- 11
正在处理 1 ----- 12
正在处理 1 ----- 13
正在处理 1 ----- 14
正在处理 1 ----- 15
正在处理 1 ----- 16
正在处理 1 ----- 17


/tmp/ipykernel_113/1190781402.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["text_tersor"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 1

正在读取第 2 个DF
初始化batch数据： 2
batch数据 组装完毕 ：2
batch_tensor数据 组装完毕 ：2,开始分块处理
分块处理完毕数据完毕，每块的长度为:2048,一共有 17 块， 开始调用模型
正在处理 2 ----- 1
正在处理 2 ----- 2
正在处理 2 ----- 3
正在处理 2 ----- 4
正在处理 2 ----- 5
正在处理 2 ----- 6
正在处理 2 ----- 7
正在处理 2 ----- 8
正在处理 2 ----- 9
正在处理 2 ----- 10
正在处理 2 ----- 11
正在处理 2 ----- 12
正在处理 2 ----- 13
正在处理 2 ----- 14
正在处理 2 ----- 15
正在处理 2 ----- 16
正在处理 2 ----- 17


/tmp/ipykernel_113/1190781402.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["text_tersor"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 2

正在读取第 3 个DF
初始化batch数据： 3
batch数据 组装完毕 ：3
batch_tensor数据 组装完毕 ：3,开始分块处理
分块处理完毕数据完毕，每块的长度为:2048,一共有 17 块， 开始调用模型
正在处理 3 ----- 1
正在处理 3 ----- 2
正在处理 3 ----- 3
正在处理 3 ----- 4
正在处理 3 ----- 5
正在处理 3 ----- 6
正在处理 3 ----- 7
正在处理 3 ----- 8
正在处理 3 ----- 9
正在处理 3 ----- 10
正在处理 3 ----- 11
正在处理 3 ----- 12
正在处理 3 ----- 13
正在处理 3 ----- 14
正在处理 3 ----- 15
正在处理 3 ----- 16
正在处理 3 ----- 17


...

文件写入完毕 38

正在读取第 39 个DF
初始化batch数据： 39
batch数据 组装完毕 ：39
batch_tensor数据 组装完毕 ：39,开始分块处理
分块处理完毕数据完毕，每块的长度为:2048,一共有 17 块， 开始调用模型
正在处理 39 ----- 1
正在处理 39 ----- 2
正在处理 39 ----- 3
正在处理 39 ----- 4
正在处理 39 ----- 5
正在处理 39 ----- 6
正在处理 39 ----- 7
正在处理 39 ----- 8
正在处理 39 ----- 9
正在处理 39 ----- 10
正在处理 39 ----- 11
正在处理 39 ----- 12
正在处理 39 ----- 13
正在处理 39 ----- 14
正在处理 39 ----- 15
正在处理 39 ----- 16
正在处理 39 ----- 17


/tmp/ipykernel_113/1190781402.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["text_tersor"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 39

正在读取第 40 个DF
初始化batch数据： 40
batch数据 组装完毕 ：40
batch_tensor数据 组装完毕 ：40,开始分块处理
分块处理完毕数据完毕，每块的长度为:2048,一共有 6 块， 开始调用模型
正在处理 40 ----- 1
正在处理 40 ----- 2
正在处理 40 ----- 3
正在处理 40 ----- 4
正在处理 40 ----- 5
正在处理 40 ----- 6


/tmp/ipykernel_113/1190781402.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["text_tersor"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 40



### 合并处理